# 05 — From the laptop to an organization

This notebook produces every cell of **Table 22** (annual footprint of a
10,000-employee firm), the per-token ratios of **Finding 8**, and the 2026–2030
projection of **Finding 9**. It performs no new measurement: it reads the CSVs
written by notebooks 01 and 02 and applies published multipliers to them.

**Multipliers and where they come from**

| Quantity | Value | Source |
|---|---|---|
| PUE (Azure) | 1.12 | Jegham et al. (2025), Table 1 — from Microsoft's Azure sustainability disclosure |
| WUE on-site | 0.30 L/kWh | idem |
| WUE source | 4.35 L/kWh | idem |
| CIF | 0.35 kgCO2e/kWh | idem — a fleet-wide average across Microsoft data centres, not one region |
| PUE (AWS) | 1.14 | Jegham et al. (2025), Table 1 — Amazon 2023 sustainability report |
| Published per-query energy | Table 4 | Jegham et al. (2025) |

Water and carbon follow equations 4 and 5 of that paper:
`Water = E/PUE × WUE_site + E × WUE_source` and `Carbon = E × CIF`.

**Two caveats stated in the thesis.** The CIF is a fleet average rather than a
regional intensity — on the French grid the same workload carries roughly a
sixth of that carbon. And the Qwen models measured here are not served on Azure:
the Azure multipliers are a worked illustration of what cloud overhead does to a
locally measured figure, not a claim about Alibaba's infrastructure.


In [ ]:
"""Table 17 - annual footprint of a 10,000-employee firm, rebuilt from the
measured CSVs and the published multipliers, so every cell is traceable.

Sources of the multipliers
  PUE 1.12, WUE_site 0.30 L/kWh, WUE_source 4.35 L/kWh, CIF 0.35 kgCO2e/kWh
      Jegham et al. (2025), Table 1, Azure row
      (themselves from Microsoft's Azure sustainability blog and Microsoft's
       2024 Environmental Sustainability Report data fact sheet)
  Published per-query energy, long prompt    Jegham et al. (2025), Table 4
  AWS PUE 1.14, used to strip the facility overhead off those figures
      Jegham et al. (2025), Table 1, AWS row (Amazon 2023 sustainability report)

Formulas, Jegham et al. (2025), eq. 4 and 5
  Carbon = E_facility x CIF
  Water  = E_facility / PUE x WUE_site + E_facility x WUE_source
"""
import pandas as pd

# ------------------------------------------------------------- workload ----
EMPLOYEES, ACTIVE, PER_DAY, DAYS = 10_000, 0.35, 8, 220
REQ = EMPLOYEES * ACTIVE * PER_DAY * DAYS            # 6,160,000 requests a year
MIX = {'T1': 0.60, 'T2': 0.30, 'T3': 0.10}

PUE, WUE_SITE, WUE_SRC, CIF = 1.12, 0.30, 4.35, 0.35
AWS_PUE = 1.14

# published per-query energy, Wh, long prompt (~1,000 output tokens)
PUB = {'qwen2.5:0.5b': ('LLaMA-3.2 1B',  0.552),
       'qwen2.5:1.5b': ('LLaMA-3.2 1B',  0.552),
       'qwen2.5:3b':   ('LLaMA-3.2 3B',  0.707),
       'qwen2.5:7b':   ('LLaMA-3.1 8B',  0.443)}
PUB_TOKENS = 1_350   # the paper's 'long' output is ~1,000 words

b = pd.read_csv('../data/benchmark_runs.csv')
wh = b.groupby(['model', 'tier']).energy_wh.mean().unstack()
tok = b.groupby(['model', 'tier']).tokens_out.mean().unstack()
mixed = lambda df, m: sum(MIX[t] * df.loc[m, t] for t in MIX)

def footprint(e_it_wh_per_req, n=REQ):
    """IT watt-hours per request -> MWh, tCO2e and m3 of water a year."""
    e_fac = e_it_wh_per_req * PUE * n / 1e6                       # MWh facility
    return (e_fac,
            e_fac * CIF,                                          # tCO2e
            (e_fac / PUE) * WUE_SITE + e_fac * WUE_SRC)           # m3

MODELS = ['qwen2.5:0.5b', 'qwen2.5:1.5b', 'qwen2.5:3b', 'qwen2.5:7b']
NAME = {m: 'Qwen2.5 ' + m.split(':')[1].upper() for m in MODELS}

# ---------------------------------------------- A. local measurement -------
print(f'requests a year: {REQ:,}\n')
print('A. this thesis, measured on the test machine, Azure multipliers applied')
print(f"{'strategy':28} {'Wh/req':>8} {'MWh':>7} {'tCO2e':>7} {'m3':>7} {'vs 7B':>8}")
locA, eA = {}, {}
for m in MODELS:
    eA[m] = mixed(wh, m); locA[m] = footprint(eA[m])
for m in MODELS:
    print(f'{NAME[m]:28} {eA[m]:8.5f} {locA[m][0]:7.3f} {locA[m][1]:7.3f} {locA[m][2]:7.2f} '
          f'{100*(locA[m][0]/locA["qwen2.5:7b"][0]-1):+7.1f}%')

# the executed cascade, reweighted onto the same 60/30/10 mix
c = pd.read_csv('../data/cascade_runs.csv')
casc = c[c.pipeline == 'cascade'].groupby('tier').energy_wh.mean()
e_c = sum(MIX[t] * casc[t] for t in MIX)
f_c = footprint(e_c)
print(f'{"Cascade, as executed":28} {e_c:8.5f} {f_c[0]:7.3f} {f_c[1]:7.3f} {f_c[2]:7.2f} '
      f'{100*(f_c[0]/locA["qwen2.5:7b"][0]-1):+7.1f}%')

# ------------------------------------------- B. published cloud figures ----
print('\nB. published cloud figures, Jegham et al. (2025) Table 4, per output token')
print(f"{'strategy':28} {'proxy':14} {'Wh/req':>8} {'MWh':>7} {'tCO2e':>7} {'vs 7B':>8}")
locB, eB = {}, {}
for m in MODELS:
    proxy, e_long = PUB[m]
    eB[m] = e_long / AWS_PUE / PUB_TOKENS * mixed(tok, m)
    locB[m] = footprint(eB[m])
for m in MODELS:
    print(f'{NAME[m]:28} {PUB[m][0]:14} {eB[m]:8.5f} {locB[m][0]:7.3f} {locB[m][1]:7.3f} '
          f'{100*(locB[m][0]/locB["qwen2.5:7b"][0]-1):+7.1f}%')

r = (locA['qwen2.5:0.5b'][0] / locA['qwen2.5:7b'][0]) / \
    (locB['qwen2.5:0.5b'][0] / locB['qwen2.5:7b'][0])
print(f'\nranking divergence, 0.5B against 7B, local over published: {1/r:.1f}x')

# ----------------------------------------- per-token ratios, Finding 8 -----
print("\nFinding 8: local Wh per output token against the published cloud figure")
for m in MODELS:
    loc = eA[m] / mixed(tok, m)
    cld = PUB[m][1] / AWS_PUE / PUB_TOKENS
    print(f'  {NAME[m]:14} local {loc:.3e}   published {cld:.3e}   ratio {loc/cld:6.3f}')
sp = [eA[m]/mixed(tok,m) / (PUB[m][1]/AWS_PUE/PUB_TOKENS) for m in MODELS]
print(f'  spread across the four models: {max(sp)/min(sp):.1f}x')
print(f'  local 0.5B/7B energy ratio      {eA["qwen2.5:0.5b"]/eA["qwen2.5:7b"]:.3f}')
print(f'  published 0.5B/7B energy ratio  {eB["qwen2.5:0.5b"]/eB["qwen2.5:7b"]:.3f}')
print(f'  carbon per active user, always-7B published: '
      f'{locB["qwen2.5:7b"][1]*1e6/(EMPLOYEES*ACTIVE):.0f} g CO2e a year')

# ------------------------------------------- 2026-2030 projection ---------
import math
G = 1.40 * 0.90                    # +40% usage, -10% energy per request a year
print(f'\n2026-2030, net compound growth {100*(G-1):.0f}% a year, factor {G**4:.2f} over four years')
for label, keep in [('no governance, always-7B', 1.0),
                    ('always-3B from 2026', locA['qwen2.5:3b'][0]/locA['qwen2.5:7b'][0]),
                    ('always-3B, half the saving reabsorbed',
                     1 - 0.5*(1 - locA['qwen2.5:3b'][0]/locA['qwen2.5:7b'][0]))]:
    y26 = locA['qwen2.5:7b'][0] * keep
    y30 = y26 * G**4
    cross = math.log(1/keep)/math.log(G) if keep < 1 else 0
    print(f'  {label:40} {y26:.3f} -> {y30:.3f} MWh   '
          f'{100*(y30/locA["qwen2.5:7b"][0]-1):+6.0f}% vs 2026 baseline'
          + (f'   crosses it after {cross:.1f} yr' if keep < 1 else ''))
print(f'\npublished per-query energy, Jegham Table 4: GPT-4o 2.875 Wh / LLaMA-3.1 8B 0.443 Wh '
      f'= {2.875/0.443:.1f}x')
